# Matrix Factorization using Singular Value Decomposition (SVD)

## Objective

In this notebook, we implement a recommendation system using **Matrix Factorization** based on **Singular Value Decomposition (SVD)**.

Unlike User-Based and Item-Based Collaborative Filtering, Matrix Factorization learns **latent (hidden) features** of users and movies.

These latent features capture underlying preferences that are not explicitly available in the dataset.

The workflow includes:

- Loading the processed dataset
- Creating the User–Movie Interaction Matrix
- Applying Truncated SVD
- Learning latent embeddings for users and movies
- Reconstructing the rating matrix
- Generating personalized recommendations

Matrix Factorization is one of the most influential techniques in recommendation systems and formed the foundation of the Netflix Prize-winning solution.

# What is Matrix Factorization?

Instead of directly comparing users or movies, Matrix Factorization compresses the large User–Movie Rating Matrix into a smaller set of hidden features called **latent factors**.

These latent factors may represent concepts such as:

- Preference for Action movies
- Preference for Comedy
- Preference for Romance
- Preference for Science Fiction

The algorithm discovers these patterns automatically from user ratings.

This allows the model to recommend movies even when two users have never watched exactly the same movies.

In [1]:
import pandas as pd
import numpy as np

from sklearn.decomposition import TruncatedSVD

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
movie_ratings = pd.read_csv(
    "processed_data/movie_ratings_processed.csv"
)

movie_ratings.head()

,user_id,movie_id,rating,timestamp,movie_title,user_encoded,movie_encoded
0,196,242,3,881250949,Kolya (1996),195,241
1,186,302,3,891717742,L.A. Confidential (1997),185,301
2,22,377,1,878887116,Heavyweights (1994),21,376
3,244,51,2,880606923,Legends of the Fall (1994),243,50
4,166,346,1,886397596,Jackie Brown (1997),165,345


# Creating the User–Movie Matrix

The User–Movie Matrix is the input to the Matrix Factorization algorithm.

Rows represent users.

Columns represent movies.

Each cell stores the rating given by a user.

Missing ratings are replaced with **0** because Truncated SVD requires a complete numerical matrix.

In [3]:
user_movie_matrix = movie_ratings.pivot_table(
    index="user_id",
    columns="movie_title",
    values="rating"
)

user_movie_matrix = user_movie_matrix.fillna(0)

print(user_movie_matrix.shape)

user_movie_matrix.head()

(943, 1664)


movie_title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,2.0,5.0,0.0,0.0,3.0,4.0,0.0,0.0,...,0.0,0.0,0.0,5.0,3.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,2.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,...,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,4.0,0.0


# Why Do We Need Matrix Factorization?

Our User–Movie Matrix contains thousands of values.

Most of them are missing because users rate only a small number of movies.

This results in a **highly sparse matrix**.

Instead of storing thousands of movie ratings, Matrix Factorization compresses each user and movie into a small vector of latent features.

For example,

Instead of representing a user with 1664 movie ratings,

we may represent the same user using only **50 latent features**.

This greatly reduces dimensionality while preserving important information.

In [4]:
svd = TruncatedSVD(
    n_components=50,
    random_state=42
)

user_embeddings = svd.fit_transform(user_movie_matrix)

print("User Embeddings Shape:")
print(user_embeddings.shape)

User Embeddings Shape:
(943, 50)


# Understanding Latent Embeddings

After applying SVD, each user is represented by a vector of latent features.

Instead of 1664 movie ratings,

each user is now represented by only 50 numerical values.

These values capture hidden preferences learned automatically from the rating data.

These latent vectors are called **Embeddings**.

In [5]:
user_embeddings_df = pd.DataFrame(user_embeddings)

user_embeddings_df.head()

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,42.043422,-1.664974,-1.504603,-13.232066,-1.373398,-2.455140,10.683289,2.048045,-6.288519,1.783210,...,-0.927891,-3.030461,-7.503306,-1.177249,2.890222,-0.253742,-2.720006,-6.574519,-4.582502,-9.237924
1,8.949614,11.332479,11.399634,2.693323,2.249586,-0.805017,2.221449,0.668281,-3.767594,-0.441966,...,1.054280,-1.255287,2.426386,-1.536785,1.911899,0.088747,-0.518464,2.282758,-0.895738,-0.966301
2,3.618921,6.239959,5.156245,3.988138,-7.171478,-0.538339,0.598842,0.324907,2.405869,-0.676893,...,-0.034776,-0.855813,-1.090589,0.809254,0.327687,-0.056035,1.243082,-0.437090,1.368236,-1.345938
3,3.847190,5.089566,2.793742,2.845738,-4.194924,1.743970,1.402905,0.965034,2.239359,0.770970,...,0.855991,0.726215,-0.170992,1.185312,0.378650,0.019840,-0.530024,2.591905,1.274696,0.838490
4,20.988305,-2.175145,-10.056875,-2.370013,-1.517414,1.975597,12.086783,-9.001514,1.365370,2.134960,...,-1.525999,-1.906463,-2.662789,-0.383496,-2.270307,1.898602,-1.900444,-1.435556,-1.241544,-0.620153


# Why are these called Embeddings?

An embedding is a dense numerical representation of an object.

Instead of representing a user using thousands of movie ratings,

we represent the same user using a small vector of meaningful numbers.

Later, in the PyTorch notebook, we will train these embeddings directly using neural networks.

## Interview Question

### Q. What is the difference between Collaborative Filtering and Matrix Factorization?

**Answer:**

Collaborative Filtering compares users or items directly using similarity measures.

Matrix Factorization learns hidden latent features of users and items, enabling it to discover deeper relationships in the data.

It generally produces more accurate recommendations and scales better to larger datasets.

In [6]:
# ==========================================
# Save User Embeddings
# ==========================================

import os

os.makedirs("models", exist_ok=True)

user_embeddings_df.to_csv(
    "models/user_embeddings.csv",
    index=False
)

print("User embeddings saved successfully!")

User embeddings saved successfully!


In [7]:
# ==========================================
# Save Item Embeddings
# ==========================================

item_embeddings_df.to_csv(
    "models/item_embeddings.csv",
    index=False
)

print("Item embeddings saved successfully!")

NameError: name 'item_embeddings_df' is not defined

In [8]:
print(type(user_embeddings_df))

<class 'pandas.DataFrame'>


In [9]:
locals().keys()

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__session__', '_i', '_ii', '_iii', '_i1', 'pd', 'np', 'TruncatedSVD', '_i2', 'movie_ratings', '_2', '_i3', 'user_movie_matrix', '_3', '_i4', 'svd', 'user_embeddings', '_i5', 'user_embeddings_df', '_5', '_i6', 'os', '_i7', '_i8', '_i9'])

In [10]:
[item for item in locals().keys() if "embed" in item.lower()]

['user_embeddings', 'user_embeddings_df']

In [11]:
import os

print(os.listdir("models"))

['user_embeddings.csv']


In [12]:
# ==========================================
# Save User ID Mapping
# ==========================================

user_mapping = pd.DataFrame({
    "user_id": user_movie_matrix.index
})

user_mapping.to_csv(
    "models/user_mapping.csv",
    index=False
)

print("User mapping saved successfully!")

User mapping saved successfully!


In [13]:
import os
print(os.listdir("models"))

['user_mapping.csv', 'user_embeddings.csv']


In [14]:
import os

print(os.listdir("models"))

['user_mapping.csv', 'user_embeddings.csv']


# Saving Item Embeddings

In addition to user embeddings, we also generate embeddings for movies.

Since Truncated SVD decomposes the interaction matrix into latent representations for both users and items, these item embeddings are useful for:

- Item-to-item similarity search
- FAISS vector indexing
- Hybrid recommendation systems
- Deep Learning models
- Fast nearest-neighbor retrieval

Both user and item embeddings will be stored in the **models/** directory for later notebooks.

In [15]:
# ==========================================
# Generate Item Embeddings
# ==========================================

item_embeddings = svd.components_.T

print("Item Embeddings Shape:")
print(item_embeddings.shape)

Item Embeddings Shape:
(1664, 50)


In [16]:
# Convert to DataFrame

item_embeddings_df = pd.DataFrame(item_embeddings)

item_embeddings_df.head()

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,0.001624,0.002698,0.000210,0.005125,0.001055,-0.006749,-0.002546,0.002526,-0.000332,-0.007288,...,-0.007650,0.000245,-0.010093,0.011674,-0.001831,0.007365,0.005276,-0.004057,-0.005334,0.009576
1,0.000682,-0.001052,0.001622,-0.004210,-0.001853,-0.000039,-0.001830,-0.004645,0.003169,-0.002253,...,-0.004007,-0.005475,0.004288,-0.009995,-0.001647,-0.000540,-0.000163,0.006481,-0.002030,0.009758
2,0.019588,0.023182,-0.022548,0.031619,0.049843,-0.005576,-0.000742,-0.039704,0.019048,0.040223,...,0.004177,-0.047635,0.013117,0.005835,0.080010,0.012464,-0.014217,-0.026449,0.024202,-0.063588
3,0.040075,-0.050164,0.027922,0.019611,0.004512,0.012933,-0.028934,-0.002981,0.013599,-0.060541,...,-0.027087,-0.017110,0.014709,-0.077057,0.045308,0.118034,-0.034136,-0.053401,-0.026346,0.019044
4,0.005678,0.017245,0.012098,0.010417,-0.020244,-0.013344,-0.005297,-0.006657,0.022161,-0.003566,...,0.009073,0.007704,0.013373,-0.000956,0.009969,0.015972,-0.011854,0.010993,-0.005069,-0.021810


In [17]:
# ==========================================
# Save Item Embeddings
# ==========================================

item_embeddings_df.to_csv(
    "models/item_embeddings.csv",
    index=False
)

print("Item embeddings saved successfully!")

Item embeddings saved successfully!


In [18]:
# ==========================================
# Save Movie Mapping
# ==========================================

item_mapping = pd.DataFrame({
    "movie_title": user_movie_matrix.columns
})

item_mapping.to_csv(
    "models/item_mapping.csv",
    index=False
)

print("Movie mapping saved successfully!")

Movie mapping saved successfully!


In [19]:
import os

print(os.listdir("models"))

['item_mapping.csv', 'item_embeddings.csv', 'user_mapping.csv', 'user_embeddings.csv']
